# Exercise 02 — Risk and return: the historical record

MSc Finance · Investments · FHNW · Autumn 2026

A century of daily U.S. market returns: what the tails look like, how badly the normal
distribution describes them, and why the answer changes with the investment horizon.

**How to work with this notebook.** The task text is here and on the exercise sheet. Each
code cell is a stub: the `# TODO` lines are the steps, in order. The setup and data cells
below are complete — run them first and leave them alone.

Run **Runtime → Restart and run all** before you trust any number in here.

**Data.** Kenneth R. French Data Library, *Fama/French 3 Factors (Daily)*, file
`F-F_Research_Data_Factors_daily.csv`, 202512 CRSP vintage. The market return used
throughout is $R = (\text{Mkt-RF}) + \text{RF}$, the total return on the CRSP
value-weighted index, in U.S. dollars. Source and documentation:
[mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from exercise_utils import FHNW, setup_style, describe, save_results
setup_style()

In [ ]:
BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_02/data/F-F_Research_Data_Factors_daily.csv"

# The file carries four lines of documentation on top and a copyright line at the
# bottom; returns are in percent and the date is an integer YYYYMMDD.
ff = pd.read_csv(DATA_URL, skiprows=4, skipfooter=2, engine="python")
ff.columns = ["date", "Mkt-RF", "SMB", "HML", "RF"]
ff["date"] = pd.to_datetime(ff["date"].astype(int).astype(str), format="%Y%m%d")
ff = ff.set_index("date").astype(float) / 100.0

R = (ff["Mkt-RF"] + ff["RF"]).rename("Market")   # total return on the market
rf = ff["RF"].rename("RF")                       # daily risk-free rate

print(f"{len(R)} trading days, {R.index.min().date()} to {R.index.max().date()}")

## Task 1 — The wealth index and the worst that ever happened

Cumulate the daily market return into a wealth index: one dollar invested on the first day
of the sample. The Fama/French series is in U.S. dollars throughout, so every level in this
exercise is a dollar amount and every return is an unhedged USD return. Find the five worst single days and express each one in standard deviations
from the daily mean. Then compute the drawdown series — the distance between the wealth
index and its own running maximum — and report the largest drawdown together with three
dates: the peak it fell from, the trough, and the day the loss was finally recovered.

Draw the wealth index and the drawdown series in one two-panel figure.

*Deliverable: the table of five worst days in σ units, the maximum drawdown with its three
dates, and the two-panel figure.*

In [ ]:
# SOLUTION
# STUB: cumulate the daily returns into a wealth index W (one USD at the start)
W = (1 + R).cumprod()

# STUB: the five worst days, in percent and in standard deviations from the mean
worst = R.nsmallest(5).to_frame("Return")
worst["Sigma"] = (worst["Return"] - R.mean()) / R.std()
print(worst.to_string(formatters={"Return": "{:7.2%}".format,
                                  "Sigma": "{:6.1f}".format}))

# STUB: drawdown = wealth index relative to its running maximum, minus one
dd = W / W.cummax() - 1

# STUB: the maximum drawdown and its peak, trough and recovery dates
trough = dd.idxmin()
peak = W.loc[:trough].idxmax()
after = W.loc[trough:]
recovery = after[after >= W.loc[peak]].index.min()

print(f"\nmaximum drawdown {dd.min():.1%}")
print(f"peak {peak.date()}   trough {trough.date()}   recovery {recovery.date()}")

In [ ]:
# SOLUTION
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True,          # KEEP
                               gridspec_kw={"height_ratios": [2, 1]})       # KEEP

# STUB: top panel — the wealth index. Think about the vertical scale before you plot it.
ax1.plot(W.index, W, color=FHNW["navy"], lw=1.0)
ax1.set_yscale("log")
ax1.set_ylabel("Wealth index, USD, log scale")

# STUB: bottom panel — the drawdown series
ax2.fill_between(dd.index, dd, 0, color=FHNW["red"], alpha=0.55, lw=0)
ax2.set_ylabel("Drawdown")
ax2.set_xlabel("Year")

fig.tight_layout()                                                          # KEEP
plt.show()                                                                  # KEEP

<!-- solution -->
One dollar invested in July 1926 grows to about 16,000 dollars by the end of 2025 — but
only on a logarithmic axis is that visible as a straight-ish line rather than a hockey
stick. On a linear axis the whole first seventy years look flat, which is why every
long-horizon wealth chart in the profession uses a log scale.

The five worst days are 19 October 1987 (−17.4 %), 16 March 2020 (−12.0 %) and three days
from the Great Crash of October and November 1929. In standard deviations they are −16.2σ,
−11.2σ, −10.7σ, −10.5σ and −9.0σ. Under a normal distribution a −16σ day has a probability
of about 1 in 10⁵⁹ — one would not expect to see one in the lifetime of the universe. We
have seen one in a hundred years.

Note the discrepancy with the slide: lecture 02 quotes −22.6 % for Black Monday. That is
the Dow Jones Industrial Average. The CRSP value-weighted index, which is what this file
contains, fell 17.4 % that day.

Maximum drawdown −84.1 %: peak 3 September 1929, trough 8 July 1932, and the peak was not
regained until 6 February 1945. Two things to take from it. First, the drawdown is
path-dependent — it depends on the *order* of the returns, unlike the mean, the variance,
or VaR, all of which are unchanged if the sample is shuffled. Second, sixteen years is
longer than most investment horizons, most careers, and every backtest anyone will show
you.

The common mistake is to compute the drawdown against the *final* maximum rather than the
running one: `W / W.max() - 1` puts the whole series under water and reports the maximum
drawdown as the distance from 2025 to 1932.

## Task 2 — How wrong is the normal distribution?

Report the mean, the volatility, the skewness and the excess kurtosis of the daily series.
Then draw the histogram of the daily returns with a fitted normal density on top — **twice,
side by side**: once with a linear density axis and once with a logarithmic one. The two
panels show the same two objects; decide for yourself which of them you would put in a risk
report.

Now put a number on that disagreement. Compute the Value at Risk and the Expected
Shortfall at the 5 % and the 1 % level, each of them twice:

- parametrically, assuming $R \sim N(\hat\mu, \hat\sigma^2)$, and
- empirically, read straight off the 26,151 observations in the sample.

Finally, compute the probability of a day worse than −5 % both ways.

*Deliverable: the two-panel histogram, a table with four tail numbers at each of the two
confidence levels, and the two shortfall probabilities.*

In [ ]:
# SOLUTION
mu, sd = R.mean(), R.std()                                                   # KEEP

# STUB: mean, volatility, skewness and excess kurtosis of the daily series
print(f"mean {mu:.4%}   volatility {sd:.4%}   "
      f"skewness {R.skew():.2f}   excess kurtosis {R.kurt():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))                            # KEEP
grid = np.linspace(-0.20, 0.20, 800)                                         # KEEP

for ax in axes:                                                              # KEEP
    # STUB: histogram of the daily returns, and the normal density with the same mu and sd
    ax.hist(R, bins=200, density=True, color=FHNW["blue"], alpha=0.75)
    ax.plot(grid, stats.norm.pdf(grid, mu, sd), color=FHNW["red"], lw=1.6)
    ax.set_xlim(-0.20, 0.20)
    ax.set_xlabel("Daily return")

# STUB: left panel linear, right panel logarithmic
axes[0].set_ylabel("Density")
axes[1].set_yscale("log")
axes[1].set_ylim(1e-2, 100)        # a single observation has density about 0.02
axes[1].set_ylabel("Density, log scale")

fig.tight_layout()                                                           # KEEP
plt.show()                                                                   # KEEP

In [ ]:
# SOLUTION
rows = []
for alpha in [0.05, 0.01]:                                                   # KEEP
    z = stats.norm.ppf(alpha)                                                # KEEP

    # STUB: VaR and ES under the normal assumption (both reported as positive losses)
    var_n = -(mu + z * sd)
    es_n = -(mu - sd * stats.norm.pdf(z) / alpha)

    # STUB: the same two numbers read directly off the sample
    var_e = -R.quantile(alpha)
    es_e = -R[R <= R.quantile(alpha)].mean()

    rows.append({"alpha": alpha, "VaR normal": var_n, "VaR empirical": var_e,
                 "ES normal": es_n, "ES empirical": es_e})

tail = pd.DataFrame(rows).set_index("alpha")
print(tail.to_string(float_format="{:6.2%}".format))

# STUB: probability of a day worse than -5 %, empirically and under normality
print(f"\nP(R < -5%)   empirical {(R < -0.05).mean():.5f}"
      f"   normal {stats.norm.cdf(-0.05, mu, sd):.9f}")

<!-- solution -->
Daily mean 0.0429 %, volatility 1.08 %, skewness −0.17, excess kurtosis 16.2. The skewness
is mild; the kurtosis is enormous. A normal distribution has excess kurtosis 0 by
construction.

The two panels are the point. On the left, the fitted normal looks like a reasonable
description: bars and curve agree around the centre, where 99 % of the observations sit,
and the disagreement in the tails is invisible because both densities are indistinguishable
from zero there. On the right, the same two objects on a logarithmic density axis: the
empirical bars stand one to twenty orders of magnitude above the red curve from about ±4 %
outwards.

Expect the right-hand panel to look wrong at first. A normal density plotted on a log axis
is an inverted parabola, not a bell — taking logs of $\exp(-(x-\mu)^2/2\sigma^2)$ leaves a
quadratic in $x$. Nothing has gone wrong with the fit; that shape *is* what a Gaussian looks
like on this scale, and the curvature is exactly what makes the disagreement legible. The
empirical distribution falls off roughly linearly in the log — much closer to exponential
tails than to Gaussian ones.

The tail table contains the result worth remembering:

| | VaR normal | VaR empirical | ES normal | ES empirical |
|---|---|---|---|---|
| 5 % | 1.73 % | 1.57 % | 2.18 % | 2.58 % |
| 1 % | 2.47 % | 3.08 % | 2.83 % | 4.51 % |

At the 5 % level the normal assumption is *conservative* — it reports a larger loss
threshold than the data. At the 1 % level it is much too optimistic, understating VaR by a
fifth and ES by more than a third. Both facts have the same cause: a leptokurtic
distribution has more mass in the centre *and* more in the far tails than a normal with the
same variance, and it pays for both with less mass in between. The 5 % quantile sits in
that thinned-out region; the 1 % quantile is already in the fat part.

The practical reading: a risk report that validates its model at the 95 % level and then
applies it at 99 % will pass its own back-test and still be wrong where it matters. Note
also that ES exceeds VaR at both levels, and that the *gap* between them is far wider
empirically than under normality — the tail is not merely longer, it is heavier all the way
out.

The shortfall probability makes the same point without any quantile machinery. A day worse
than −5 % happened 67 times in 26,151 days, once every 390 trading days or so. Under
normality its probability is 1.45 in a million: once every 2,700 years.

## Task 3 — Where the tails come from, and where they go

Fat tails are not a property that returns simply have. They are produced by volatility that
changes over time: a mixture of quiet periods and violent ones has fatter tails than any
single normal distribution, even when every individual period is normal.

First look at the volatility itself. Plot the rolling 252-day standard deviation,
annualised, and mark the full-sample volatility as a horizontal reference. Report the
lowest, the highest and the median value of the rolling series.

Then aggregate. Use the `to_frequency()` function in the next cell to compound the daily
returns to monthly and annual returns, and use `describe()` from `exercise_utils` to report
the number of observations, the annualised mean and volatility, the skewness, the excess
kurtosis and the Sharpe ratio at each of the three frequencies.

Two columns of that table behave completely differently as the horizon lengthens: the
excess kurtosis and the Sharpe ratio. Say which of the two is roughly horizon-invariant,
and why the other is not.

*Deliverable: the rolling-volatility figure with its three numbers, the three-row moments
table, and one sentence contrasting the two columns.*

In [ ]:
# SOLUTION
# STUB: rolling 252-day standard deviation, annualised
vol = R.rolling(252).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(9, 3.6))                                     # KEEP

# STUB: plot it, with the full-sample annualised volatility as a horizontal line
ax.plot(vol.index, vol, color=FHNW["navy"], lw=0.9)
ax.axhline(R.std() * np.sqrt(252), color=FHNW["orange"], lw=1.2, ls="--")
ax.set_ylabel("Annualised volatility")
ax.set_xlabel("Year")

plt.show()                                                                   # KEEP

# STUB: lowest, highest and median of the rolling series
print(f"lowest  {vol.min():.1%}  ({vol.idxmin().date()})")
print(f"highest {vol.max():.1%}  ({vol.idxmax().date()})")
print(f"median  {vol.median():.1%}")

In [ ]:
def to_frequency(x, rule):
    # Compound a daily return series into lower-frequency returns.
    #
    # rule: "ME" month end, "YE" year end, "QE" quarter end (pandas 2.2 and later;
    # the older aliases "M", "A", "Q" are deprecated).
    #
    # Returns compound, they do not add up — hence prod(), never sum().
    return (1 + x).resample(rule).prod() - 1


PERIODS = {"Daily": 252, "Monthly": 12, "Annual": 1}
RULES = {"Daily": None, "Monthly": "ME", "Annual": "YE"}

In [ ]:
# SOLUTION
rows = {}
for label, rule in RULES.items():                                            # KEEP
    x = R if rule is None else to_frequency(R, rule)                         # KEEP
    y = rf if rule is None else to_frequency(rf, rule)                       # KEEP

    # STUB: describe() the market at this frequency, annualising with PERIODS[label]
    tbl = describe(pd.concat([x, y], axis=1), rf_col="RF", periods=PERIODS[label])
    rows[label] = pd.concat([pd.Series({"Obs": len(x)}), tbl.loc["Market"]])

moments = pd.DataFrame(rows).T
print(moments.to_string(float_format="{:8.3f}".format))

<!-- solution -->
The rolling volatility runs from 4.8 % (November 1964) to 46.1 % (March 1933) around a
median of 12.6 % — a factor of ten between the calmest and the most violent year in the
sample. It is also visibly persistent: high-volatility days arrive next to other
high-volatility days, in episodes lasting months, not scattered at random. That clustering
is the mechanism. Pooling a 5 % year and a 45 % year into one sample and fitting a single
normal produces exactly the shape task 2 found: too much mass in the middle, too much in
the tails, too little in between.

The moments table across the three frequencies:

| | Obs | Arith. mean | Geo. mean | Volatility | Skewness | Excess kurtosis | Sharpe |
|---|---|---|---|---|---|---|---|
| Daily | 26,151 | 0.108 | 0.098 | 0.171 | −0.166 | 16.15 | 0.393 |
| Monthly | 1,194 | 0.115 | 0.102 | 0.184 | 0.115 | 7.37 | 0.383 |
| Annual | 100 | 0.121 | 0.102 | 0.197 | −0.488 | 0.04 | 0.356 |

Everything except the observation count is annualised, which is why the three means agree to
within a percentage point and the geometric mean is the same 10.2 % at every horizon — it
has to be, it is the same terminal wealth divided up differently.

The excess kurtosis is 16.1 daily, 7.4 monthly, 0.04 annual; in raw kurtosis, 19.1 → 10.4 →
3.0, the numbers from the lecture. At the annual horizon the market is statistically
indistinguishable from a normal distribution. Aggregation averages over the volatility
regimes, and a sum of many draws tends to normality — the central limit theorem doing its
work on the very mixture that created the fat tails in the first place.

Now the contrast the task asks for. The Sharpe ratio is 0.39, 0.38, 0.36 — essentially the
same number at every horizon. The excess kurtosis falls by a factor of four hundred over
the same range. The first two moments annualise cleanly, by scaling rules that hold exactly
under independence and nearly enough in practice; the third and fourth do not, because
aggregation is precisely what averages away the volatility variation that produced them.
That asymmetry is the reason portfolio theory is built on the mean and the variance and
almost never on anything higher, and it is also the reason a risk model calibrated on daily
data cannot be carried to an annual horizon by multiplying by √252.

One detail worth knowing: `pandas.kurt()` applies a small-sample correction and
`scipy.stats.kurtosis()` — which is what `describe()` calls — does not. At n = 100 the two
differ in the second decimal; at n = 26,151 they agree.

## Export

Bundle the figures and the tables you produced, in case you want them for the transfer
questions or your own notes.

In [ ]:
# SOLUTION
# STUB: export the moments and tail tables together with one figure
save_results(figures={"rolling_volatility": fig},
             tables={"moments": moments, "tail_risk": tail},
             name="ex02")

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Lecture 03 takes the two moments that survived this exercise intact — the mean and the
volatility — and asks how much of your wealth belongs in the risky asset at all.